<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day15_practice1_%ED%86%A0%ED%81%B0%ED%99%94_%EC%9D%B8%EC%BD%94%EB%94%A9_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 텍스트를 숫자로 - 토큰화와 인코딩

In [2]:
import torch

In [3]:
# 코퍼스 - 텍스트 데이터의 집합
corpus = [
    "이 영화 정말 재미있다",
    "정말 최고의 영화",
    "이 영화 너무 지루하다",
    "최악이다 정말 지루하다",
]

In [6]:
sent = corpus[0]
print("원문:", sent)
print("1. 공백(어절) 단위:", sent.split())
print("2. 문자 단위      :", list(sent.replace(" ","")))

원문: 이 영화 정말 재미있다
1. 공백(어절) 단위: ['이', '영화', '정말', '재미있다']
2. 문자 단위      : ['이', '영', '화', '정', '말', '재', '미', '있', '다']


In [8]:
# 셀 2. 단어 사전(vocab)
# 신경망에 넣으려면 각 단어를 '고유 번호'로 바꿔야 한다.
# 특수 토큰 2개
# <pad>=0 : 길이 맞추기용 빈칸
# <unk>=1 : 사전에 없는 단어 (unknown)

vocab = {"<pad>": 0, "<unk>": 1} # dict (단어: 번호)
for s in corpus:
  for tck in s.split():
    if tck not in vocab:
      vocab[tck] = len(vocab)

print("\n단어 사전:", vocab)
print("사전 크기:", len(vocab))


단어 사전: {'<pad>': 0, '<unk>': 1, '이': 2, '영화': 3, '정말': 4, '재미있다': 5, '최고의': 6, '너무': 7, '지루하다': 8, '최악이다': 9}
사전 크기: 10


In [10]:
# 셀 3. 정수 인코딩
# 단사전을 이용해 문장 전체를 '번호 리스트'로 변환하는 게 인코딩,
def encode(sentence):
  return [vocab.get(tck, vocab["<unk>"]) for tck in sentence.split()]

for s in corpus:
  print(f"{s:30s} -> {encode(s)}")

print("\n처음 보는 단어 포함:", "이 영화 완전 최고".split(),"->", encode("이 영화 완전 최고"), "('완전'은 <unk>=1)")

이 영화 정말 재미있다                   -> [2, 3, 4, 5]
정말 최고의 영화                      -> [4, 6, 3]
이 영화 너무 지루하다                   -> [2, 3, 7, 8]
최악이다 정말 지루하다                   -> [9, 4, 8]

처음 보는 단어 포함: ['이', '영화', '완전', '최고'] -> [2, 3, 1, 1] ('완전'은 <unk>=1)


In [12]:
# 셀 4. 패딩
# DataLoader 는 '같은 모양' 텐서만 묶는다. 짧은 문장 위를 <pad>(0)로 채워 길이를 통일한다.
encoded = [encode(s) for s in corpus]
max_len = max(len(e) for e in encoded)
padded = torch.tensor([e + [0] * (max_len - len(e)) for e in encoded]) # 각 문장 위에 부족한 만큼 0을 이어붙임
print("\n 패딩 후 (모두 길이", max_len, ")")
print(padded)


 패딩 후 (모두 길이 4 )
tensor([[2, 3, 4, 5],
        [4, 6, 3, 0],
        [2, 3, 7, 8],
        [9, 4, 8, 0]])


In [17]:
# 셀 5. 원-핫의 두 가지 문제
onehot = torch.nn.functional.one_hot(torch.tensor(encode(sent)), num_classes=len(vocab)).float()
print("\n '이 영화 정말 재미있다'의 원-핫 모양:", tuple(onehot.shape))

# 문제 1 : 차원 폭발: 실전 사전은 수만-수십만 단어
# 문제 2 : 의미가 없다 : 모든 단어 쌍의 거리가 똑같다
# 해결책 : 임베딩

v재미 = onehot[3]
v최고 = torch.nn.functional.one_hot(torch.tensor(vocab["최고의"]), len(vocab)).float()
v지루 = torch.nn.functional.one_hot(torch.tensor(vocab["지루하다"]), len(vocab)).float()
print(f" '재미있다' · '최고의' 내적 = {v재미 @ v최고:.0f}")
print(f" '재미있다' · '지루하다' 내적 = {v재미 @ v지루:.0f}")


 '이 영화 정말 재미있다'의 원-핫 모양: (4, 10)
 '재미있다' · '최고의' 내적 = 0
 '재미있다' · '지루하다' 내적 = 0
